# EcoFOCIpy PostQC example to update netcdf file

see [EcoFOCIpy_sbe_UAF.ipynb](EcoFOCIpy_sbe_UAF.ipynb) for initial processing and formatting.  Data would have been convertted from raw instrument format to csv files, initial dirty plots created and a meta-data rich and meta-data poor dataset would have been created.  The metadata poor csv file is used for the initial preliminary data archive, the metadata rich netcdf file is used for QC procedures and eventual final data archive.

## Next Steps

QC of data (plot parameters with other instruments)
- be sure to updated the qc_status and the history

- **TODO** Programmatically simplify following tools

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import yaml

import pandas as pd
import xarray as xr
import datetime
import EcoFOCIpy.io.erddap as erddap

import EcoFOCIpy.metaconfig.load_config as load_config

/Users/bell/src/ecofocipy/src/EcoFOCIpy/__init__.py:62: UserWarning: Could not import all EcoFOCIpy submodules: cannot import name 'instruments' from partially initialized module 'EcoFOCIpy' (most likely due to a circular import) (/Users/bell/src/ecofocipy/src/EcoFOCIpy/__init__.py)
  warnings.warn(f"Could not import all EcoFOCIpy submodules: {e}")


In [2]:
sample_data_dir = '/Users/bell/Programs/EcoFOCIpy/'
user_data_dir = '/Users/bell/ecoraid/2025/Moorings/25ckp2a/working/'
filename = '25ckp2a_s37_0036m.nc'

### Post Manual QC load data and rebuild nc file

- using excel for a few points or a dynamic web map for multiple (a tool on ecofoci-field.pmel.noaa.gov) or any other method to identify spikes
- remove values above 32psu and below 30psu

**NOTE** if you use excel, be cautious about how your time is formatted (its important) - use custom formatting and make it of the form `yyyy-mm-ddTHH:MM:SSZ` to assure proper read in to xarray

*TODO:* use erddap hosted data 

In [3]:
# this loads the initial working netcdf file
wop_nc_qc = xr.load_dataset(user_data_dir+filename)

In [4]:
wop_nc_qc

<xarray.Dataset> Size: 296kB
Dimensions:       (time: 9252, depth: 1, latitude: 1, longitude: 1)
Coordinates:
  * time          (time) datetime64[ns] 74kB 2025-08-21T06:00:00 ... 2026-09-...
  * depth         (depth) float64 8B 36.0
  * latitude      (latitude) float64 8B 71.22
  * longitude     (longitude) float64 8B -164.2
Data variables:
    temperature   (time, depth, latitude, longitude) float64 74kB -1.321 ... ...
    conductivity  (time, depth, latitude, longitude) float64 74kB 2.599 ... 2...
    salinity      (time, depth, latitude, longitude) float64 74kB 32.34 ... 3...
Attributes: (12/33)
    Latitude_DegMMddW:                      71 12.937 N
    Longitude_DegMMddN:                     164 14.9404 W
    MooringID:                              25CKP-2A
    platform_deployment_date:               2025-08-21T05:40:01Z
    platform_deployment_cruise_name:        SKQ202515S
    platform_recovery_date:                 2026-09-10T17:28:46Z
    ...                                     ...
    InstrumentSerialNumber:                 1805
    InstrumentType:                         SBE-37
    date_created:                           2026-10-09T07:18:55Z
    date_modified:                          
    QC_indicator:                           unknown
    history:                                Trimmed to deployment.

In [5]:
# this reads the previous csv and assumes you've made modifications to the content but not the structure (record length or variable names)
df_qc = pd.read_csv(user_data_dir+filename.replace('nc','to_edit.csv'), index_col=['time','depth','latitude','longitude']) #order is important

In [6]:
df_qc

,,,,temperature,conductivity,salinity
time,depth,latitude,longitude,,,
2025-08-21T06:00:00Z,36,71.215617,-164.249007,-1.3209,2.59891,32.3367
2025-08-21T07:00:00Z,36,71.215617,-164.249007,-1.3162,2.59920,32.3355
2025-08-21T08:00:00Z,36,71.215617,-164.249007,-1.2929,2.60096,32.3344
2025-08-21T09:00:00Z,36,71.215617,-164.249007,-1.2918,2.60106,32.3346
2025-08-21T10:00:00Z,36,71.215617,-164.249007,-1.2572,2.60354,32.3310
...,...,...,...,...,...,...
2026-09-10T13:00:00Z,36,71.215617,-164.249007,-0.5608,2.66179,32.3694
2026-09-10T14:00:00Z,36,71.215617,-164.249007,-0.5401,2.66316,32.3656
2026-09-10T15:00:00Z,36,71.215617,-164.249007,-0.5105,2.66511,32.3601


In [7]:
#replace original data with editied data
for var in wop_nc_qc.variables:
    if var not in ['time','latitude','longitude','depth'] + []:
        print(var)
        try:
            wop_nc_qc[var].values=xr.Dataset.from_dataframe(df_qc)[var]
            print(f'{var} success')
        except:
            pass

temperature
temperature success
conductivity
conductivity success
salinity
salinity success


In [8]:
# for var in ['latitude','longitude','depth']:
#     print(var)
#     try:
#         wop_nc_qc.coords[var]=xr.Dataset.from_dataframe(df_qc)[var]
#         print(f'{var} success')
#     except:
#         pass

In [9]:
#drop oxy conc for oxygen
# wop_nc_qc = wop_nc_qc.drop('oxy_conc')

### Update global attributes

In [10]:
wop_nc_qc.attrs.update({'QC_indicator': 'ProbablyGood'})
try:
    wop_nc_qc.attrs.update({'history':(wop_nc_qc.history + "\nQC'd: "+ str(datetime.datetime.today()))})
except:
    pass

In [11]:
wop_nc_qc.attrs.update({'date_modified':str(datetime.datetime.today())})

In [12]:
wop_nc_qc.to_netcdf(user_data_dir+filename,format="NETCDF3_CLASSIC",encoding={'time':{'units':'days since 1900-01-01'}})

/var/folders/6r/n_mpcj7d7pdf9ncqvjy5vb8c0000jm/T/ipykernel_50871/664318075.py:1: UserWarning: Times can't be serialized faithfully to int64 with requested units 'days since 1900-01-01'. Resolution of 'hours' needed. Serializing times to floating point instead. Set encoding['dtype'] to integer dtype to serialize to int64. Set encoding['dtype'] to floating point dtype to silence this warning.
  wop_nc_qc.to_netcdf(user_data_dir+filename,format="NETCDF3_CLASSIC",encoding={'time':{'units':'days since 1900-01-01'}})


In [13]:
wop_nc_qc

<xarray.Dataset> Size: 296kB
Dimensions:       (time: 9252, depth: 1, latitude: 1, longitude: 1)
Coordinates:
  * time          (time) datetime64[ns] 74kB 2025-08-21T06:00:00 ... 2026-09-...
  * depth         (depth) float64 8B 36.0
  * latitude      (latitude) float64 8B 71.22
  * longitude     (longitude) float64 8B -164.2
Data variables:
    temperature   (time, depth, latitude, longitude) float64 74kB -1.321 ... ...
    conductivity  (time, depth, latitude, longitude) float64 74kB 2.599 ... 2...
    salinity      (time, depth, latitude, longitude) float64 74kB 32.34 ... 3...
Attributes: (12/33)
    Latitude_DegMMddW:                      71 12.937 N
    Longitude_DegMMddN:                     164 14.9404 W
    MooringID:                              25CKP-2A
    platform_deployment_date:               2025-08-21T05:40:01Z
    platform_deployment_cruise_name:        SKQ202515S
    platform_recovery_date:                 2026-09-10T17:28:46Z
    ...                                     ...
    InstrumentSerialNumber:                 1805
    InstrumentType:                         SBE-37
    date_created:                           2026-10-09T07:18:55Z
    date_modified:                          2026-11-02 08:39:56.088017
    QC_indicator:                           ProbablyGood
    history:                                Trimmed to deployment.\nQC'd: 202...